Modelo BERTopic sobre todos los chunks
--------------------------------------

In [1]:
import os, pickle, json
from pathlib import Path
import pandas as pd
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction import text
import spacy
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
import numpy as np



# Cargar Datos


In [7]:
# Leer chunks, está en excel
chunks_etiquetados = pd.read_excel("../data/results/chunks_etiquetados_binario.xlsx")
chunks_etiquetados.head()

,chunk_id,id_doc,texto_chunk,Ciencias_ambientales_ingenieria,Ciencias_espacio,Ciencias_fisicas,Ciencias_Geografia_oceanografia,Ciencias_medicas,Ciencias_metereologia,Ciencias_naturales,...,Ciencias_tierra,Ciencia_Administracion_ciencia_investigacion,Ciencia_biologia,Ciencia_enfoque_cientifico,Ciencia_hidrologia,Ciencia_matematicas_estadistica,Ciencia_patologia,Ciencia_recursos_naturales,categorias_detectadas,etiqueta_ciencia
0,0,1,"La Coalición Colombia Partido Alianza Verde, P...",0.411230,0.368818,0.363447,0.354240,0.378154,0.343397,0.387251,...,0.392998,0.423331,0.337017,0.352550,0.328395,0.394268,0.393815,0.389038,"[('ninguna', 0)]",0
1,1,1,"al mismo tiempo lo exponen, en ciertas ocasion...",0.331604,0.304309,0.346682,0.315253,0.344347,0.323262,0.344607,...,0.318409,0.378231,0.317467,0.368125,0.306375,0.381964,0.337864,0.306510,"[('ninguna', 0)]",0
2,2,1,los acuerdos con las Farc. Anunció que no prom...,0.356805,0.311163,0.303825,0.287517,0.329738,0.290927,0.332603,...,0.292858,0.405040,0.291387,0.322885,0.297822,0.338846,0.294842,0.318955,"[('ninguna', 0)]",0
3,3,1,moratoria en la explotación tipo fracking. Y f...,0.435032,0.397881,0.372960,0.368983,0.415538,0.357000,0.410173,...,0.392092,0.445708,0.370653,0.367256,0.341832,0.404120,0.375238,0.397525,"[('ninguna', 0)]",0
4,0,2,Las interpretaciones de la historia sirven com...,0.352643,0.367811,0.375234,0.357391,0.368184,0.347770,0.392265,...,0.368177,0.428733,0.353389,0.405911,0.371109,0.412153,0.397781,0.339795,"[('ninguna', 0)]",0


In [8]:
# Verifico contenido
len(chunks_etiquetados)

62651

In [9]:
# Leer embeddings gemma
embeddings = np.load(r"../data/processed/chunk_embeddings.npy")

In [10]:
print(type(embeddings))  
print(embeddings.shape)  # (n_documentos, dim_embedding)

<class 'numpy.ndarray'>
(62651, 768)


## Stopwords

In [11]:
# Cargar modelo pequeño de spaCy (rápido)
nlp = spacy.load("es_core_news_sm", disable=["parser", "ner"])

# Stopwords en español desde spaCy
spanish_stopwords = nlp.Defaults.stop_words

# BERTopic

In [57]:
documents = chunks_etiquetados['texto_chunk'].tolist()

In [58]:
vectorizer_model = CountVectorizer(
    stop_words= list(spanish_stopwords),
    max_features=5000,   
    min_df=5,
    ngram_range=(1, 1),
    strip_accents=None
)

In [ ]:
topic_model = BERTopic(
    vectorizer_model=vectorizer_model,
    nr_topics=10,
    min_topic_size=20,
    calculate_probabilities=False,
    verbose=True,
    language="spanish"
)

In [80]:
topics, probs = topic_model.fit_transform(
    documents,
    embeddings=embeddings   # embeddings Gemma
)

2026-01-21 23:45:51,622 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-01-21 23:46:10,380 - BERTopic - Dimensionality - Completed ✓
2026-01-21 23:46:10,386 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-01-21 23:46:18,211 - BERTopic - Cluster - Completed ✓
2026-01-21 23:46:18,213 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-01-21 23:46:26,595 - BERTopic - Representation - Completed ✓
2026-01-21 23:46:26,604 - BERTopic - Topic reduction - Reducing number of topics
2026-01-21 23:46:26,761 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-01-21 23:46:34,149 - BERTopic - Representation - Completed ✓
2026-01-21 23:46:34,165 - BERTopic - Topic reduction - Reduced number of topics from 240 to 10


In [99]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,37252,-1_paz_gobierno_política_país,"[paz, gobierno, política, país, justicia, urib...",[aunque es probable que se plantee una nueva n...
1,0,1848,0_educación_estudiantes_universidades_universidad,"[educación, estudiantes, universidades, univer...",[educación básica y la superior. El puente est...
2,1,1111,1_fútbol_jugadores_equipo_copa,"[fútbol, jugadores, equipo, copa, equipos, gol...",[y en posición rematadora Martínez y Carmelo. ...
3,2,1044,2_libros_libro_novela_literatura,"[libros, libro, novela, literatura, escritor, ...",[descubrí que era un libro de cuentos cortos. ...
4,3,1026,3_maduro_venezuela_venezolanos_guaidó,"[maduro, venezuela, venezolanos, guaidó, venez...","[ClaverCarone, director de Asuntos del Hemisfe..."
...,...,...,...,...,...
844,843,5,843_belleza_velocidad_cielo_vicios,"[belleza, velocidad, cielo, vicios, dientes, s...",[Todos los días salimos a la calle. Todos los ...
845,844,5,844_monstruo_desconocer_emprender_buscar,"[monstruo, desconocer, emprender, buscar, firm...","[ello, actuamos frente al proceso con el ELN b..."
846,845,5,845_contraloría_00_regalías_contratación,"[contraloría, 00, regalías, contratación, 0005...",[percibe la reforma de la Contraloría que estu...
847,846,5,846_defiende_talante_1998_aspiraciones,"[defiende, talante, 1998, aspiraciones, discur...",[Horacio Serpa la tuvo en 1998; Vargas Lleras ...


# Coherence con Gensim


In [82]:
# Extraemos el analizador del vectorizador
analyzer = vectorizer_model.build_analyzer()

# Tokenizamos todos los documentos
tokenized_docs = [analyzer(doc) for doc in documents]

In [83]:
# Diccionario gensim
dictionary = Dictionary(tokenized_docs)

# Filtramos extremos para estabilidad (opcional pero recomendado)
dictionary.filter_extremes(no_below=5, no_above=0.9)

# Corpus Bag-of-Words
corpus = [dictionary.doc2bow(doc) for doc in tokenized_docs]

In [84]:
def get_topic_words(topic_model, top_n=10):
    topics_words = []

    for topic_id, word_scores in topic_model.get_topics().items():
        if topic_id == -1:
            continue  # excluimos outliers

        words = [word for word, _ in word_scores[:top_n]]
        topics_words.append(words)

    return topics_words


In [85]:
topics_words = get_topic_words(topic_model, top_n=10)


In [86]:
len(topics_words)

9

In [87]:
# Calcular coherencia global del modelo
coherence_model = CoherenceModel(
    topics=topics_words,
    texts=tokenized_docs,
    dictionary=dictionary,
    corpus=corpus,
    coherence='c_v'
)

coherence_score = coherence_model.get_coherence()
coherence_score


0.41506608495888336

In [88]:
# coherencia por tópico para detectar tópicos débiles

topic_coherences = coherence_model.get_coherence_per_topic()

for i, score in enumerate(topic_coherences):
    print(f"Tópico {i}: coherencia = {score:.4f}")


Tópico 0: coherencia = 0.3714
Tópico 1: coherencia = 0.3459
Tópico 2: coherencia = 0.4107
Tópico 3: coherencia = 0.3783
Tópico 4: coherencia = 0.4679
Tópico 5: coherencia = 0.4047
Tópico 6: coherencia = 0.4931
Tópico 7: coherencia = 0.3602
Tópico 8: coherencia = 0.5034


In [89]:
sum(score > 0.5 for score in topic_coherences) / len(topic_coherences)


0.1111111111111111

In [90]:
# Función que junta ambos métodos

def compute_bertopic_coherence(
    topic_model,
    documents,
    vectorizer_model,
    top_n_words=10
):
    analyzer = vectorizer_model.build_analyzer()
    tokenized_docs = [analyzer(doc) for doc in documents]

    dictionary = Dictionary(tokenized_docs)
    dictionary.filter_extremes(no_below=5, no_above=0.9)

    corpus = [dictionary.doc2bow(doc) for doc in tokenized_docs]

    topics_words = get_topic_words(topic_model, top_n=top_n_words)

    coherence_model = CoherenceModel(
        topics=topics_words,
        texts=tokenized_docs,
        dictionary=dictionary,
        corpus=corpus,
        coherence='c_v'
    )

    return coherence_model.get_coherence(), coherence_model.get_coherence_per_topic()


# Reducir outliers

In [91]:
new_topics  = topic_model.reduce_outliers(
    documents,
    topics,
    strategy="c-tf-idf",
    embeddings=embeddings
    )

topic_model.update_topics(documents, topics=new_topics, vectorizer_model=vectorizer_model)

2026-01-21 23:47:43,213 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


In [98]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,37252,-1_paz_gobierno_política_país,"[paz, gobierno, política, país, justicia, urib...",[aunque es probable que se plantee una nueva n...
1,0,1848,0_educación_estudiantes_universidades_universidad,"[educación, estudiantes, universidades, univer...",[educación básica y la superior. El puente est...
2,1,1111,1_fútbol_jugadores_equipo_copa,"[fútbol, jugadores, equipo, copa, equipos, gol...",[y en posición rematadora Martínez y Carmelo. ...
3,2,1044,2_libros_libro_novela_literatura,"[libros, libro, novela, literatura, escritor, ...",[descubrí que era un libro de cuentos cortos. ...
4,3,1026,3_maduro_venezuela_venezolanos_guaidó,"[maduro, venezuela, venezolanos, guaidó, venez...","[ClaverCarone, director de Asuntos del Hemisfe..."
...,...,...,...,...,...
844,843,5,843_belleza_velocidad_cielo_vicios,"[belleza, velocidad, cielo, vicios, dientes, s...",[Todos los días salimos a la calle. Todos los ...
845,844,5,844_monstruo_desconocer_emprender_buscar,"[monstruo, desconocer, emprender, buscar, firm...","[ello, actuamos frente al proceso con el ELN b..."
846,845,5,845_contraloría_00_regalías_contratación,"[contraloría, 00, regalías, contratación, 0005...",[percibe la reforma de la Contraloría que estu...
847,846,5,846_defiende_talante_1998_aspiraciones,"[defiende, talante, 1998, aspiraciones, discur...",[Horacio Serpa la tuvo en 1998; Vargas Lleras ...


In [93]:
coherence_reduced,_ = compute_bertopic_coherence(topic_model, documents, vectorizer_model)

In [96]:
coherence_reduced

0.4566389203017137

In [97]:
fig = topic_model.visualize_barchart(
    top_n_topics=20,    # ajusta según tu experimento
    n_words=10
)
fig.show()

# Probando varios valores de número de tópicos

In [100]:
nr_topics_values = [10, 20, 30, 40, 60, "auto"]
results = []

for nr in nr_topics_values:
    topic_model = BERTopic(
        vectorizer_model=vectorizer_model,
        nr_topics=nr,
        min_topic_size=20,
        calculate_probabilities=False,
        verbose=True,
        language="spanish"
    )

    topics, probs = topic_model.fit_transform(
        documents,
        embeddings=embeddings   # embeddings Gemma
    )

    new_topics  = topic_model.reduce_outliers(
        documents,
        topics,
        strategy="c-tf-idf",
        embeddings=embeddings
        )

    topic_model.update_topics(documents, topics=new_topics, vectorizer_model=vectorizer_model)

    coherence,_ = compute_bertopic_coherence(topic_model, documents, vectorizer_model)

    n_outliers = sum(1 for t in topic_model.topics_ if t == -1)


    results.append({
        "nr_topics": nr,
        "n_topics_final": len([t for t in topic_model.get_topics() if t != -1]),
        "coherence_c_v": coherence,
        "%_outliers": n_outliers / len(documents)
    })


2026-01-21 23:54:27,575 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-01-21 23:55:15,460 - BERTopic - Dimensionality - Completed ✓
2026-01-21 23:55:15,467 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-01-21 23:55:31,069 - BERTopic - Cluster - Completed ✓
2026-01-21 23:55:31,071 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-01-21 23:55:45,579 - BERTopic - Representation - Completed ✓
2026-01-21 23:55:45,590 - BERTopic - Topic reduction - Reducing number of topics
2026-01-21 23:55:46,026 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-01-21 23:56:00,541 - BERTopic - Representation - Completed ✓
2026-01-21 23:56:00,572 - BERTopic - Topic reduction - Reduced number of topics from 106 to 10
2026-01-21 23:56:08,502 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure th

In [101]:
df_results = pd.DataFrame(results)
df_results.sort_values("coherence_c_v", ascending=False)


,nr_topics,n_topics_final,coherence_c_v,%_outliers
5,auto,63,0.681289,0.0
4,60,59,0.671076,0.0
2,30,29,0.652175,0.0
3,40,39,0.635335,0.0
1,20,19,0.545472,0.0
0,10,9,0.480771,0.0


In [ ]:
tm_res = topic_model.get_topic_info()

In [108]:
tm_res.head(60)

,Topic,Count,Name,Representation,Representative_Docs
0,0,25096,0_presidente_gobierno_duque_paz,"[presidente, gobierno, duque, paz, país, uribe...",[ver con los proyectos de país y los intereses...
1,1,3300,1_pandemia_virus_salud_coronavirus,"[pandemia, virus, salud, coronavirus, covid19,...",[Tengo 60 años recién cumplidos y no recuerdo ...
2,2,3494,2_economía_impuestos_crecimiento_ingresos,"[economía, impuestos, crecimiento, ingresos, f...",[del salario mínimo ocasionaría la ampliación ...
3,3,2660,3_educación_estudiantes_universidad_universidades,"[educación, estudiantes, universidad, universi...",[educación básica y la superior. El puente est...
4,4,2095,4_mujeres_mujer_hombres_sexual,"[mujeres, mujer, hombres, sexual, violencia, g...",[de los delitos contra la libertad y la integr...
5,5,1336,5_fútbol_jugadores_equipo_juego,"[fútbol, jugadores, equipo, juego, equipos, co...",[que también seguramente hubo quien lo pensara...
6,6,1009,6_cocina_comida_comer_restaurante,"[cocina, comida, comer, restaurante, plato, sa...","[y servicio. Pero lo más destacable, incluso p..."
7,7,1585,7_redes_información_datos_facebook,"[redes, información, datos, facebook, medios, ...",[Me imagino que de aquí al domingo los colombi...
8,8,1396,8_hectáreas_rural_deforestación_tierras,"[hectáreas, rural, deforestación, tierras, agr...",[el establecimiento de 280.000 hectáreas de si...
9,9,877,9_ciudad_bogotá_metro_buses,"[ciudad, bogotá, metro, buses, transporte, tra...",[día más competitivos los precios de los vehíc...


In [119]:
coh_global, coh_per_topic = compute_bertopic_coherence(
    topic_model=topic_model,
    documents=documents,
    vectorizer_model=vectorizer_model,
    top_n_words=10
)
topic_ids = [
    topic_id
    for topic_id in topic_model.get_topics().keys()
    if topic_id != -1
]
coherence_df = pd.DataFrame({
    "Topic": topic_ids,
    "Coherence_c_v": coh_per_topic
})

In [121]:
coherence_table = (
    tm_res[tm_res["Topic"] != -1]
    .merge(coherence_df, on="Topic", how="left")
)

In [122]:
total_docs = tm_res["Count"].sum()


In [110]:
top = (
    tm_res
    .sort_values("Count", ascending=False)
    .head(10)
    .copy()
)
top["Keywords"] = top["Representation"].apply(
    lambda words: ", ".join(words)
)


In [111]:
top["Percentage"] = (top["Count"] / total_docs * 100).round(2)


In [123]:
top = top.merge(
    coherence_df,
    on="Topic",
    how="left"
)


In [127]:
final_table = top[[
    "Topic",
    #"Count",
    "Coherence_c_v",
    "Percentage",
    "Keywords"
]]

final_table

,Topic,Coherence_c_v,Percentage,Keywords
0,0,0.550938,40.06,"presidente, gobierno, duque, paz, país, uribe, colombia, política, años, petro"
1,2,0.843713,5.58,"economía, impuestos, crecimiento, ingresos, fiscal, empresas, tasa, empleo, pib, déficit"
2,1,0.732233,5.27,"pandemia, virus, salud, coronavirus, covid19, mundo, cuarentena, medidas, crisis, personas"
3,3,0.818489,4.25,"educación, estudiantes, universidad, universidades, calidad, formación, superior, nacional, profesores, colegios"
4,4,0.842738,3.34,"mujeres, mujer, hombres, sexual, violencia, género, sexo, feminismo, acoso, sexuales"
5,7,0.689015,2.53,"redes, información, datos, facebook, medios, sociales, comunicación, personas, internet, twitter"
6,8,0.787886,2.23,"hectáreas, rural, deforestación, tierras, agricultura, productores, agropecuario, cultivos, 000, desarrollo"
7,5,0.864152,2.13,"fútbol, jugadores, equipo, juego, equipos, copa, selección, deporte, gol, mundial"
8,20,0.599038,2.05,"corrupción, corruptos, públicos, consulta, anticorrupción, lucha, funcionarios, recursos, país, política"
9,6,0.899797,1.61,"cocina, comida, comer, restaurante, plato, sabores, arroz, platos, sabor, casa"


In [128]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 20)
pd.set_option("display.width", None)

final_table

,Topic,Coherence_c_v,Percentage,Keywords
0,0,0.550938,40.06,"presidente, gobierno, duque, paz, país, uribe, colombia, política, años, petro"
1,2,0.843713,5.58,"economía, impuestos, crecimiento, ingresos, fiscal, empresas, tasa, empleo, pib, déficit"
2,1,0.732233,5.27,"pandemia, virus, salud, coronavirus, covid19, mundo, cuarentena, medidas, crisis, personas"
3,3,0.818489,4.25,"educación, estudiantes, universidad, universidades, calidad, formación, superior, nacional, profesores, colegios"
4,4,0.842738,3.34,"mujeres, mujer, hombres, sexual, violencia, género, sexo, feminismo, acoso, sexuales"
5,7,0.689015,2.53,"redes, información, datos, facebook, medios, sociales, comunicación, personas, internet, twitter"
6,8,0.787886,2.23,"hectáreas, rural, deforestación, tierras, agricultura, productores, agropecuario, cultivos, 000, desarrollo"
7,5,0.864152,2.13,"fútbol, jugadores, equipo, juego, equipos, copa, selección, deporte, gol, mundial"
8,20,0.599038,2.05,"corrupción, corruptos, públicos, consulta, anticorrupción, lucha, funcionarios, recursos, país, política"
9,6,0.899797,1.61,"cocina, comida, comer, restaurante, plato, sabores, arroz, platos, sabor, casa"
